<a href="https://colab.research.google.com/github/sravand2001/guild-mle-projects/blob/main/Copy_of_Student_MLE_MiniProject_Deep_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scikeras

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.metrics import RocCurveDisplay
from keras.models import Sequential
from keras.layers import Dense, Input
from scikeras.wrappers import KerasClassifier
from sklearn.pipeline import Pipeline

In [ ]:
DATA_PATH = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'

# Download the dataset and load it into a pandas DataFrame
columns = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
]

df = pd.read_csv(DATA_PATH, names=columns)

In [ ]:
# Display the first few rows of the DataFrame
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [ ]:
# Do some exploratory analysis. How many rows/columns are there? How are NULL
# values represented? What's the percentrage of positive cases in the dataset?
print(df.shape)
print(df.isnull().sum())

positive_count = 0

for value in df['income']:
    if value.strip() == '>50K':
        positive_count += 1
print(positive_count)
positive_rate = (positive_count / len(df)) * 100

print(f'Rate = {round(positive_rate)}%')

(32561, 15)
age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
dtype: int64
7841
Rate = 24%


In [ ]:
# Find all NULL values and drop them
df = df.dropna()

In [ ]:
# Use Scikit-Learn's LabelEncoder to convert the income column with a data type
# string to a binary variable.
le = LabelEncoder()
le.fit(df['income'])
print(le.transform(df['income']))

[0 0 0 ... 0 0 1]


In [ ]:
# Split dataset into training and test sets
x = df.drop('income', axis=1)
y = df['income']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
y = le.transform(df['income'])

counts = np.bincount(y)
majority_class = np.argmax(counts)
print("Majority class:", majority_class)

method = [majority_class] * len(y)

auc = roc_auc_score(y, method)
print("AUC score:", auc)

Majority class: 0
AUC score: 0.5


In [ ]:
# Use Scikit-Learn's ColumnTransformer to apply One Hot Encoding to the
# categorical variables in workclass, education, marital-status, occupation,
# relationship, 'race', sex, and native-country. #Also, apply MinMaxScaler to
# the remaining continuous features.
cat_cols = ['workclass', 'education', 'marital-status',
                    'occupation', 'relationship', 'race', 'sex', 'native-country']

num_cols = [col for col in df.columns if col not in cat_cols]
num_cols.remove('income')

transformer = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(), cat_cols),
        ('num', MinMaxScaler(), num_cols)
    ]
)
x = df.drop('income', axis=1)
x_transformed = transformer.fit_transform(x)


In [ ]:
# How many columns will the dataframe have after these columns transformations are applied?
print("Total columns after transformation:", x_transformed.shape[1])

Total columns after transformation: 108


In [ ]:
# Define the Keras model
from scikeras.wrappers import KerasClassifier

def create_model(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(16, activation='relu'),
        Dense(8, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
# Create a Keras classifier
# Create a temporary transformer to calculate input_dim without fitting the main transformer
temp_transformer = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(), cat_cols),
        ('num', MinMaxScaler(), num_cols)
    ]
)
input_dim = temp_transformer.fit_transform(x_train).shape[1]
classifier = KerasClassifier(
    model=create_model,
    model__input_dim=input_dim,  # передаём аргумент функции create_model
    epochs=5,
    batch_size=32,
    verbose=1
)

In [ ]:
# Create the scikit-learn pipeline
pipeline = Pipeline([
    ('preprocessor', transformer),
    ('classifier', classifier)
])

In [ ]:
# Fit the pipeline on the training data
y_train_encoded = le.transform(y_train)
pipeline.fit(x_train, y_train_encoded)
keras_history = pipeline['classifier'].history_

Epoch 1/5
814/814 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8194 - loss: 0.3829
Epoch 2/5
814/814 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8409 - loss: 0.3390
Epoch 3/5
814/814 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8458 - loss: 0.3306
Epoch 4/5
814/814 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8492 - loss: 0.3242
Epoch 5/5
814/814 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8504 - loss: 0.3201


In [ ]:
x_test_transformed = pipeline['preprocessor'].transform(x_test)
predictions = pipeline['classifier'].predict_proba(x_test_transformed)[:, 1]

204/204 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


In [ ]:
# Generate an ROC curve for your model.
auc = roc_auc_score(y_test, predictions)

print("AUC:", auc)

AUC: 0.9074565404265548
